In [1]:
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, TrainingArguments, AutoModelForSequenceClassification
from datasets import Dataset
from utils import WeightedTrainer, compute_metrics


train_df = pd.read_parquet('parquets/avito_train.parquet')
val_df = pd.read_parquet('parquets/avito_val.parquet')
test_df = pd.read_parquet('parquets/avito_test.parquet')

**Level 2: BERT**

In [2]:
# Тк BERT работает с числами, а не строками, то нужно преобразовать строки в числа. Классы в числа = mapping 

child_encoder = LabelEncoder()
train_df['child_id'] = child_encoder.fit_transform(train_df['category_name'])
val_df['child_id'] = child_encoder.transform(val_df['category_name'])
test_df['child_id'] = child_encoder.transform(test_df['category_name'])

parent_encoder = LabelEncoder()
train_df['parent_id'] = parent_encoder.fit_transform(train_df['parent_category_name'])
val_df['parent_id'] = parent_encoder.transform(val_df['parent_category_name'])
test_df['parent_id'] = parent_encoder.transform(test_df['parent_category_name'])

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(
    'cointegrated/rubert-tiny2',
    num_labels=47
)

class_weights = torch.tensor(
    compute_class_weight('balanced', classes=np.arange(47), y=train_df['child_id'].values),
    dtype=torch.float
)

tokenizer = AutoTokenizer.from_pretrained('cointegrated/rubert-tiny2')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_ds = Dataset.from_pandas(train_df[['text', 'child_id']]).rename_column('child_id', 'labels').map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df[['text', 'child_id']]).rename_column('child_id', 'labels').map(tokenize, batched=True)

args = TrainingArguments(
    output_dir='./rubert_tiny2',
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none'
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/240000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18750, training_loss=0.8237767081705729, metrics={'train_runtime': 4242.3695, 'train_samples_per_second': 282.861, 'train_steps_per_second': 4.42, 'total_flos': 4450485657600000.0, 'train_loss': 0.8237767081705729, 'epoch': 5.0})

In [9]:
# dataset, тк bert не может работать с dataframe 
test_ds = (Dataset.from_pandas(test_df[['text', 'child_id']])
           .rename_column('child_id', 'labels')
           .map(tokenize, batched=True))

# predict
predictions = trainer.predict(test_ds)
preds = predictions.predictions
preds = preds.argmax(axis=-1)

# score
test_f1_macro = f1_score(test_df['child_id'], preds, average='macro')
print(f'Test F1-macro: {test_f1_macro:.4f}')

# save results
test_df['bert_pred'] = preds
test_df.to_parquet('parquets/bert_test_preds.parquet', index=False)
df = pd.read_parquet('parquets/bert_test_preds.parquet')
print(df.columns.tolist())
print(df.head(2))

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Test F1-macro: 0.8477
['item_id', 'user_id', 'region', 'city', 'parent_category_name', 'category_name', 'param_1', 'param_2', 'param_3', 'title', 'description', 'price', 'item_seq_number', 'activation_date', 'user_type', 'image', 'image_top_1', 'deal_probability', 'text', 'child_id', 'parent_id', 'bert_pred']
        item_id       user_id                   region         city  \
0  9ecc831dcd04  e1e72da64f21       Краснодарский край      Армавир   
1  2cdcd8a35e22  4b5dc92bc9b1  Калининградская область  Калининград   

  parent_category_name     category_name param_1 param_2 param_3  \
0      Для дома и дачи  Продукты питания     NaN     NaN     NaN   
1             Животные            Собаки  Другая     NaN     NaN   

                                        title  ... item_seq_number  \
0  Домашняя копченая грудинка, колбаса, сало!  ...              25   
1                                Супер собака  ...            1256   

   activation_date  user_type  \
0       2017-03-23    Priv